# TRIBE v2 Brain Activity Prediction

This notebook demonstrates how to use OBI-ONE to predict fMRI brain activity from naturalistic stimuli using [Meta's TRIBE v2](https://ai.meta.com/blog/tribe-v2-brain-predictive-foundation-model/) multimodal brain encoding model.

TRIBE v2 combines three foundation models:
- **V-JEPA2** for video
- **Wav2Vec-BERT** for audio
- **LLaMA 3.2** for text

...to predict whole-brain fMRI responses (~20,484 cortical vertices on fsaverage5) at 1 Hz.

### Prerequisites

Install tribev2 with plotting support:
```bash
pip install "tribev2[plotting] @ git+https://github.com/facebookresearch/tribev2.git"
```

You may also need to authenticate with HuggingFace for LLaMA 3.2 access:
```bash
huggingface-cli login
```

## 1. Predict brain responses to a video

In [1]:
import obi_one as obi
from pathlib import Path

OUTPUT_ROOT = Path("../../../obi-output/tribe_brain_prediction")

First, download a sample video (Sintel trailer from Blender Foundation).

In [2]:
from tribev2.demo_utils import download_file

video_dir = Path("./data")
video_dir.mkdir(exist_ok=True)
video_path = video_dir / "sintel_trailer.mp4"

if not video_path.exists():
    download_file(
        "https://download.blender.org/durian/trailer/sintel_trailer-480p.mp4",
        video_path,
    )

/Users/james/Documents/obi/code/obi-main/obi-one/.venv/lib/python3.12/site-packages/neuralset/extractors/base.py:707: UserWarning: LabelEncoder: event_types has not been set, are you sure you want to apply this extractor to all events?
  warnings.warn(
2026-04-01 12:46:37 - WARNING - neuralset.extractors.base:798 - Missing events will be encoded using the default all-zero value (for example, 0 or a zero vector/tensor), which may be indistinguishable from a valid class if that class is also mapped to zeros. Set treat_missing_as_separate_class=True to avoid this.


[2026-04-01 12:46:37,320] WARNING: Missing events will be encoded using the default all-zero value (for example, 0 or a zero vector/tensor), which may be indistinguishable from a valid class if that class is also mapped to zeros. Set treat_missing_as_separate_class=True to avoid this.


### Configure and run the prediction task

The `Stimulus` block accepts exactly one of:
- `video_path` — video file (audio + speech are extracted automatically)
- `audio_path` — audio file
- `text` — raw text (converted to speech internally)

The `Visualization` block controls the output: brain surface colormap, view angle, number of timesteps, etc.

In [ ]:
video_config = obi.TribeBrainPredictionScanConfig(
    stimulus=obi.TribeBrainPredictionScanConfig.Stimulus(
        video_path=video_path,
        start_time=0,
        end_time=10,
    ),
    model=obi.TribeBrainPredictionScanConfig.Model(
        checkpoint="facebook/tribev2",
        device="auto",
        config_update={
            "data.text_feature.model_name": "openai-community/gpt2",
            "data.text_feature.device": "cpu",
            "data.audio_feature.device": "cpu",
            "data.video_feature.image.device": "cpu",
            "accelerator": "cpu",
        },
    ),
    visualization=obi.TribeBrainPredictionScanConfig.Visualization(
        n_timesteps=15,
        cmap="fire",
        views="left",
        show_stimuli=True,
        output_video=True,
        output_timesteps_image=True,
        interpolated_fps=10,
    ),
)

grid_scan = obi.GridScanGenerationTask(
    form=video_config,
    output_root=OUTPUT_ROOT / "video",
)
grid_scan.execute()
obi.run_tasks_for_generated_scan(grid_scan)

### Inspect outputs

The task produces:
- `predictions.npy` — raw brain activity array `(n_timesteps, 20484)`
- `brain_activity_timesteps.png` — static mosaic showing brain activity per timestep
- `brain_activity.mp4` — video of brain activity over time
- `events.csv` — stimulus events (video frames, extracted words, audio segments)
- `prediction_metadata.json` — shape info, model checkpoint used, etc.

In [ ]:
import json
import numpy as np

output_dir = grid_scan.single_configs[0].coordinate_output_root

# Load and inspect metadata
with open(output_dir / "prediction_metadata.json") as f:
    metadata = json.load(f)
print("Metadata:", json.dumps(metadata, indent=2))

# Load predictions
preds = np.load(output_dir / "predictions.npy")
print(f"\nPredictions shape: {preds.shape}  (n_timesteps, n_vertices)")

### View the timesteps image

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(output_dir / "brain_activity_timesteps.png"), width=1200))

### Play the brain activity video

In [ ]:
from IPython.display import Video

Video(str(output_dir / "brain_activity.mp4"), embed=True, width=600)

## 2. Predict brain responses to text

TRIBE v2 can predict brain activity from raw text. The text is converted to speech (gTTS), then transcribed to get word-level timings.

In [ ]:
text_config = obi.TribeBrainPredictionScanConfig(
    stimulus=obi.TribeBrainPredictionScanConfig.Stimulus(
        text=(
            "To be or not to be, that is the question. "
            "Whether 'tis nobler in the mind to suffer "
            "the slings and arrows of outrageous fortune, "
            "or to take arms against a sea of troubles "
            "and by opposing end them."
        ),
    ),
    visualization=obi.TribeBrainPredictionScanConfig.Visualization(
        n_timesteps=15,
        show_stimuli=True,
        output_video=False,
    ),
)

grid_scan_text = obi.GridScanGenerationTask(
    form=text_config,
    output_root=OUTPUT_ROOT / "text",
)
grid_scan_text.execute()
obi.run_tasks_for_generated_scan(grid_scan_text)

In [ ]:
text_output_dir = grid_scan_text.single_configs[0].coordinate_output_root
display(Image(filename=str(text_output_dir / "brain_activity_timesteps.png"), width=1200))

## 3. Parameter scan: compare brain responses across multiple texts

Since `text` accepts a list, OBI-ONE's scan framework automatically generates one prediction per text passage.

In [ ]:
scan_config = obi.TribeBrainPredictionScanConfig(
    stimulus=obi.TribeBrainPredictionScanConfig.Stimulus(
        text=[
            "The quick brown fox jumps over the lazy dog.",
            "A spectre is haunting Europe, the spectre of communism.",
            "In the beginning God created the heaven and the earth.",
        ],
    ),
    visualization=obi.TribeBrainPredictionScanConfig.Visualization(
        output_video=False,
        output_timesteps_image=True,
        show_stimuli=True,
    ),
)

grid_scan_multi = obi.GridScanGenerationTask(
    form=scan_config,
    output_root=OUTPUT_ROOT / "text_scan",
)
grid_scan_multi.execute()
obi.run_tasks_for_generated_scan(grid_scan_multi)

print(f"Generated {len(grid_scan_multi.single_configs)} predictions")

### Compare predictions across texts

In [ ]:
for i, single_config in enumerate(grid_scan_multi.single_configs):
    out = single_config.coordinate_output_root
    preds = np.load(out / "predictions.npy")
    print(f"\nText {i + 1}: '{single_config.stimulus.text[:50]}...'")
    print(f"  Predictions shape: {preds.shape}")
    print(f"  Mean activation: {preds.mean():.4f}, Max: {preds.max():.4f}")
    display(Image(filename=str(out / "brain_activity_timesteps.png"), width=1000))

## 4. Advanced: custom visualization with tribev2 directly

You can also use the tribev2 plotting API directly on the saved predictions for more control.

In [ ]:
from tribev2.plotting import PlotBrain

# Load predictions from the video example
video_output = grid_scan.single_configs[0].coordinate_output_root
preds = np.load(video_output / "predictions.npy")

plotter = PlotBrain(mesh="fsaverage5")

# Plot a single timepoint from multiple views
fig = plotter.plot_surf(
    preds[10],
    views=["left", "right", "dorsal"],
    cmap="fire",
    norm_percentile=99,
)

In [ ]:
# RGB overlay: compare three timepoints as red/green/blue channels
if preds.shape[0] >= 3:
    plotter.plot_surf_rgb(
        [preds[5], preds[10], preds[15] if preds.shape[0] > 15 else preds[-1]],
        views=["left", "right"],
        cmap="rgb",
        norm_percentile=95,
    )